# 1. 인스타그램 크롤링 

### 1.1 인스타그램 접속

In [19]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import urllib.request
import time

# 크롬 드라이버 자동 업데이트
from webdriver_manager.chrome import ChromeDriverManager

# 브라우저 꺼짐 방지
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)

# 불필요한 에러 메시지 없애기
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
service = Service(executable_path=ChromeDriverManager().install())
driver = webdriver.Chrome(options=chrome_options)

In [20]:
# 인스타그램 접속하기
driver.get("https://instagram.com")
time.sleep(3)

* `인스타 로그인`을 위해 계정 `아이디` / `비번` 입력이 필요합니다. 

In [21]:
#인스타그램 자동로그인
driver.implicitly_wait(10) 
login_id = driver.find_element(By.CSS_SELECTOR, 'input[name="email"]')
login_id.send_keys('아이디')
login_pwd = driver.find_element(By.CSS_SELECTOR, 'input[name="pass"]')
login_pwd.send_keys('비밀번호')
driver.implicitly_wait(10)
login_pwd.submit()

# time.sleep(2)
# login_id.send_keys(Keys.ENTER) 

### 1.2 인스타그램 검색결과 URL 만들어서 접속하기

In [ ]:
#import time
# 인스타그램 접속하기
#driver.get('https://www.instargram.com')
#time.sleep(2)

In [30]:
def insta_searching(word):
    url = 'https://www.instagram.com/explore/search/keyword/?q=' + word
    return url

In [31]:
word = "울릉도"
url = insta_searching(word)
driver.get(url)

In [32]:
word = "제주도맛집"
url = insta_searching(word)
driver.get(url)

### 1.3 첫 번째 게시글 열기 

In [35]:
def select_first(driver):
    first = driver.find_element(By.CSS_SELECTOR, 'a[href*="/p/"]')
    first.click()
    time.sleep(5)

select_first(driver)

### 1.4 게시글 정보 가져오기 

In [37]:
# 게시글 정보 가져오기
import re
from bs4 import BeautifulSoup
import unicodedata
def get_content(driver):
    # 현재 페이지 html 정보 가져오기
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    # 본문 내용 가져오기
    try:
        content = soup.select('div._a9zs')[0].text
    except:
        content = ' '
    tags = re.findall(r'#[^\s#,\\]+', content)   
    # 작성일자 정보 가져오기 
    date = soup.select('time[datetime]')[0]['datetime'][:10]
   # 수집한 정보 저장하기
    data = [content, date, tags]
    return data
get_content(driver)

[' ', '2021-08-24', []]

### 1.5 다음 게시글 열기

In [38]:
def move_next(driver):
    right = driver.find_element(By.CSS_SELECTOR, 'div._aaqg._aaqh')
    right.click()
    time.sleep(3)
move_next(driver)

### 1.6 정보 수집하기

In [39]:
word = "제주도맛집"   #검색어
url = insta_searching(word)
driver.get(url)
time.sleep(7)
select_first(driver) #첫 번째 게시글 열기

results = []
target = 10
for i in range(target):
    try:
        data = get_content(driver) 
        results.append(data)
        move_next(driver)
    except:
        time.sleep(5)
        move_next(driver)
        
print(results[:2])  

[[' ', '2021-08-24', []], [' ', '2025-10-29', []]]


### 1.7 수집한 정보 저장하기

In [41]:
import pandas as pd
results_df = pd.DataFrame(results)
results_df.columns = ['content', 'data', 'tags']
results_df.to_excel('../data/jejudo_crawing.xlsx')

In [42]:
driver.close()

### 1.8 여러 엑셀 파일의 중복을 제거한 후 통합 저장


In [43]:
jeju_insta_df = pd.DataFrame( [ ] )

folder = '../data/'
f_list = ['1_crawling_jejudoMatJip.xlsx', '1_crawling_jejudoGwanGwang.xlsx',
          '1_crawling_jejuMatJip.xlsx', '1_crawling_jejuYeoHang.xlsx']
for fname in f_list:
    fpath = folder + fname
    temp = pd.read_excel(fpath)
    jeju_insta_df = jeju_insta_df.append(temp)

jeju_insta_df.columns =['content','data','like','place','tags']

FileNotFoundError: [Errno 2] No such file or directory: '../data/1_crawling_jejudoMatJip.xlsx'

In [43]:
jeju_insta_df.drop_duplicates(subset = [ "content"] , inplace = True)
jeju_insta_df.to_excel('../data/1_crawling_raw.xlsx', index = False)